# 2D → 3D Pipeline — Colab 1-Click Runbook (P6)

Chạy toàn bộ hệ thống trên **Google Colab Free (T4)** và mở Web UI từ máy tính qua **Cloudflare Tunnel**.

Thứ tự: **Runtime ▸ Change runtime type ▸ T4 GPU** → Run all (Cell 1 → Cell 6).

| Cell | Việc |
| :--: | :--- |
| 1 | Clone repo (branch test) + cài dependencies + **DUSt3R thật cho P2** |
| 2 | Clone TripoSR (chỉ dùng cho chế độ 1 ảnh / nhánh Quality FAIL) |
| 3 | (Tùy chọn) Upload ảnh test |
| 4 | Bật FastAPI + Cloudflare Tunnel → in ra URL công khai (lần đầu chờ 3–8 phút vì tải weights) |
| 5 | Smoke test API bằng `curl` |
| 6 | Dừng server |

> **P2 đã chạy DUSt3R thật**: hình dạng 3D lấy từ ảnh. Nếu Cell 1 báo `⚠️ DUSt3R chưa import được`
> thì P2 rơi về MOCK (hình dạng NGẪU NHIÊN) — kiểm tra lại Cell 1 trước khi chụp kết quả.


In [ ]:
# Cell 1: Clone repo (branch test) + cài dependencies + DUSt3R thật cho P2
# Colab đã có sẵn torch/torchvision GPU — KHÔNG cài lại, cài lại dễ hỏng CUDA runtime.
import os, shutil, sys

# BẮT BUỘC: kernel có thể đang đứng trong thư mục đã bị xoá (từ lần chạy trước)
os.chdir('/content')

REPO = '/content/Img2d-to-3d'
if not os.path.isdir(REPO + '/.git'):
    shutil.rmtree(REPO, ignore_errors=True)
    !git clone -q --branch P6-FullStack-Cloud https://github.com/dduy26/Img2d-to-3d.git /content/Img2d-to-3d
else:
    !git -C /content/Img2d-to-3d fetch -q origin P6-FullStack-Cloud
    !git -C /content/Img2d-to-3d checkout -q P6-FullStack-Cloud
    !git -C /content/Img2d-to-3d pull -q

%cd /content/Img2d-to-3d
!git log --oneline -1
!ls notebook/backend/app.py

# ── Cài đặt NumPy & dependencies chuẩn (tránh lỗi '_slice' C-extension & xung đột RAPIDS) ──
!pip install -q "numpy>=2.1.0,<2.3.0" "scipy>=1.14.0"
!pip install -q fastapi uvicorn python-multipart trimesh rembg onnxruntime opencv-python-headless kornia xatlas "scikit-image<0.26.0"

# ── DUSt3R THẬT (P2) — hình dạng 3D lấy từ ảnh thay vì dữ liệu ngẫu nhiên ──
if not os.path.isdir('/content/dust3r/.git'):
    shutil.rmtree('/content/dust3r', ignore_errors=True)
    !git clone -q --recursive https://github.com/naver/dust3r.git /content/dust3r
# Chỉ cài phần thực sự còn thiếu, không đè torch hay các gói có sẵn:
!pip install -q roma einops

# Nạp cả dust3r và croco submodule vào sys.path
for p in ['/content/dust3r', '/content/dust3r/croco']:
    if p not in sys.path:
        sys.path.insert(0, p)

# Bắt lỗi & Chẩn đoán tính tương thích NumPy C-extension
has_slice = False
try:
    from numpy._core.umath import _slice
    has_slice = True
except Exception as e:
    print('\n' + '='*78)
    print('🔍 [BẮT LỖI CHẨN ĐOÁN MÔI TRƯỜNG COLAB]:')
    print('❌ Lỗi:', e)
    print('📌 NGUYÊN NHÂN TẠI SAO BỊ:')
    print('   - Khi mở Colab, tiến trình Python đã nạp sẵn module C của NumPy cũ vào RAM.')
    print('   - Lệnh pip install vừa cài NumPy mới lên đĩa, nhưng RAM của Colab chưa được làm mới.')
    print('   - File umath.py mới đòi hỏi hàm _slice mà module C trong RAM chưa có.')
    print('👉 CÁCH KHẮC PHỤC NGAY:')
    print('   - Bấm vào menu: Runtime (Thời gian chạy) -> Restart session (Khởi động lại phiên làm việc)')
    print('   - Sau đó chạy lại Cell 1 này: Lỗi sẽ biến mất 100% vì RAM được nạp bản NumPy mới!')
    print('='*78 + '\n')

try:
    from dust3r.inference import inference
    from dust3r.model import AsymmetricCroCo3DStereo
    from dust3r.cloud_opt import global_aligner
    print('✅ DUSt3R & CroCo import THÀNH CÔNG -> P2 chạy THẬT (hình dạng lấy từ ảnh)')
except Exception as e:
    import traceback
    print('❌ LỖI KHI IMPORT DUST3R:')
    traceback.print_exc()
    if not has_slice:
        print('👉 Vui lòng Restart session như hướng dẫn ở trên để DUSt3R hoạt động!')
    else:
        print('⚠️ Pipeline đa ảnh sẽ bị lỗi nếu thiếu DUSt3R thật.')

# ── RMBG-2.0 (tách nền) — repo GATED, cần token HuggingFace ──
import os as _os
_tok = _os.environ.get('HF_TOKEN')
if not _tok:
    try:
        from google.colab import userdata
        _tok = userdata.get('HF_TOKEN')
        _os.environ['HF_TOKEN'] = _tok
    except Exception as e:
        _tok = None
        print('ℹ️ Chưa có secret HF_TOKEN (' + type(e).__name__ + ')')
if _tok:
    print('✅ HF_TOKEN đã nạp -> P1 tách nền thật bằng RMBG-2.0')
else:
    print('ℹ️ Không có HF_TOKEN -> P1 dùng rembg (u2net) offline. Vẫn chạy tốt.')
print('Deps OK')


In [ ]:
# Cell 2: TripoSR (chỉ cần cho chế độ 1 ảnh / nhánh Quality FAIL) — luồng đa ảnh KHÔNG cần
import os, sys

os.chdir('/content')   # kernel có thể đang đứng trong thư mục đã bị xoá -> tránh 'getcwd' fail
REPO = '/content/Img2d-to-3d'
assert os.path.isdir(REPO + '/notebook/backend'), 'Cell 1 chua chay xong -> chay lai Cell 1 truoc'

if not os.path.isdir('/content/TripoSR/.git'):
    os.system('rm -rf /content/TripoSR')
    !git clone -q https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR

# KHÔNG chạy requirements.txt của TripoSR: nó ghim transformers==4.35.0 (đè bản RMBG-2.0 cần)
!pip install -q omegaconf einops imageio
# torchmcubes: Colab giờ là Python 3.13 -> wheel cp311 vô dụng, để fail rồi TripoSR tự CPU fallback
!pip install -q torchmcubes 2>/dev/null || echo 'torchmcubes thiếu -> TripoSR tự chạy CPU fallback'
print('Python', sys.version.split()[0])

!rm -rf /content/Img2d-to-3d/notebook/backend/tsr
!cp -r /content/TripoSR/tsr /content/Img2d-to-3d/notebook/backend/tsr

os.chdir(REPO)   # TRẢ cwd về repo — Cell 4 dùng os.getcwd() nên không được để nó ở /content
print('TripoSR source OK | cwd =', os.getcwd())


In [ ]:
# Cell 3 (TÙY CHỌN): Upload ảnh test. Ảnh sẽ được lưu vào thư mục 'input' duy nhất
from google.colab import files
import os, shutil
up = files.upload()  # Chọn các file ảnh .jpg/.png của vật thể (ví dụ 4-8 góc)
INPUT_DIR = '/content/Img2d-to-3d/input'
os.makedirs(INPUT_DIR, exist_ok=True)
for name in up:
    shutil.move(name, os.path.join(INPUT_DIR, name))
print('Danh sách ảnh trong thư mục input:', sorted(os.listdir(INPUT_DIR)))


In [ ]:
# Cell 4: Bật FastAPI + Cloudflare Tunnel (KHÔNG cần tài khoản Cloudflare)
import subprocess, time, os, sys

# Dọn dẹp tiến trình cũ nếu còn chạy ngầm để không nghẽn cổng 8000
!pkill -f uvicorn 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(1)

REPO = '/content/Img2d-to-3d'
BACKEND = os.path.join(REPO, 'notebook', 'backend')
assert os.path.isdir(BACKEND), f'Khong thay {BACKEND} -> chay lai Cell 1'
os.chdir(REPO)

# Pre-flight DUSt3R check trong chính kernel này
for p in ['/content/dust3r', '/content/dust3r/croco']:
    if p not in sys.path:
        sys.path.insert(0, p)
try:
    from dust3r.inference import inference
    from dust3r.model import AsymmetricCroCo3DStereo
    print('Pre-flight check: DUSt3R import OK')
except Exception as e:
    print('Pre-flight check CẢNH BÁO: DUSt3R không import được:', e)
    if '_slice' in str(e):
        print('👉 HÃY BẤM: Runtime -> Restart session rồi chạy lại từ Cell 1!')

if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# PYTHONPATH truyền cho tiến trình con uvicorn
env = dict(os.environ)
env['PYTHONPATH'] = (
    '/content/dust3r' + os.pathsep +
    '/content/dust3r/croco' + os.pathsep +
    BACKEND + os.pathsep +
    env.get('PYTHONPATH', '')
)
if os.environ.get('HF_TOKEN'):
    env['HF_TOKEN'] = os.environ['HF_TOKEN']
env.setdefault('TSDF_RES', '128')
env.setdefault('DUST3R_NITER', '300')

server = subprocess.Popen(
    ['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=BACKEND, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

import urllib.request, json as _json
ready = False
for i in range(150):
    # BẮT LỖI TỨC THÌ: Nếu uvicorn sập thì dừng ngay lập tức, không để người dùng chờ 5 phút vô vọng
    if server.poll() is not None:
        print(f'❌ TIẾN TRÌNH UVICORN ĐÃ SẬP ĐỘT NGỘT (Mã thoát: {server.returncode})')
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/api/health', timeout=2) as r:
            if r.status == 200:
                data = r.read().decode()
                print('Server READY:', data)
                h = _json.loads(data)
                if h.get('engines', {}).get('dust3r') is True:
                    print('✅ DUSt3R Engine: THẬT (Model weights đã nạp sẵn sàng)')
                else:
                    print('❌ CẢNH BÁO: DUSt3R Engine CHƯA nạp được model thật!')
                ready = True
                break
    except Exception:
        time.sleep(2)

if not ready:
    print('\n' + '='*78)
    print('❌ [BẮT LỖI SERVER]: Server FastAPI không thể khởi động!')
    print('📋 DƯỚI ĐÂY LÀ TOÀN BỘ LOG LỖI CỦA TIẾN TRÌNH UVICORN:')
    print('='*78)
    server.terminate()
    log_err = server.stdout.read()
    print(log_err if log_err else '(Không có log output nào)')
    print('='*78 + '\n')
    raise RuntimeError('Uvicorn server failed to start! Vui lòng đọc log lỗi chi tiết ở trên.')

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
import re
url = None
for line in tunnel.stdout:
    m = re.search(r'https://[\w.-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        print('\n🌐 WEB UI  :', url)
        print('📘 SWAGGER :', url + '/docs')
        break
if url is None:
    print('Không lấy được URL tunnel — xem log trên. Thử lại Cell 4.')


In [ ]:
# Cell 5: Smoke test API qua cURL với các ảnh trong thư mục input
import glob, subprocess, json, os

INPUT_DIR = '/content/Img2d-to-3d/input'
exts = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
imgs = []
for ext in exts:
    imgs.extend(glob.glob(os.path.join(INPUT_DIR, ext)))
imgs = sorted(list(set(imgs)))
print(f'Tìm thấy {len(imgs)} ảnh trong thư mục input:', [os.path.basename(p) for p in imgs])
assert imgs, 'Không có ảnh nào trong thư mục input! Vui lòng upload ảnh ở Cell 3 hoặc qua Web UI.'

if len(imgs) >= 2:
    cmd = ['curl', '-s', '-X', 'POST', 'http://127.0.0.1:8000/generate-3d/']
    for p in imgs:
        cmd += ['-F', f'files=@{p}']
else:
    cmd = ['curl', '-s', '-X', 'POST', 'http://127.0.0.1:8000/generate-3d/single/', '-F', f'file=@{imgs[0]}']

r = subprocess.run(cmd, capture_output=True, text=True)
out = r.stdout
print(json.dumps(json.loads(out), indent=2, ensure_ascii=False) if out else f'Không có phản hồi. stderr={r.stderr[-500:]}')


In [ ]:
# Cell 6: Dừng server + tunnel khi xong
tunnel.terminate(); server.terminate()
print('Đã dừng. File .glb nằm ở thư mục output (/content/Img2d-to-3d/output/)')
